# SSL pretraining demo: one EchoNet-LVH video sample

Walk through one video end to end: raw video → clip sampling → augmentation → dataset output → SimCLR forward pass + loss.

Run with the project `.venv` kernel; the notebook expects to be run from inside `ssl_pretraining/`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision

In [ ]:
SSL_DIR = Path.cwd() / "ssl_pretraining"
sys.path.insert(0, str(SSL_DIR))

from dataset import EchoNetLVHDataset
from losses import NT_Xent
from model import SimCLR
from utils import load_video, set_seed


DATA_DIR = SSL_DIR.parent / "data"
DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
set_seed(0)
print(f"data_dir={DATA_DIR}  device={DEVICE}  torch={torch.__version__}")


def show_clip(clip, title=""):
    """Plot the frames of a (T, H, W, C) clip side by side."""
    fig, axes = plt.subplots(1, len(clip), figsize=(3 * len(clip), 3))
    for t, ax in enumerate(np.atleast_1d(axes)):
        frame = clip[t]
        ax.imshow(frame if frame.dtype == np.uint8 else np.clip(frame, 0, 1))
        ax.set_title(f"frame {t}")
        ax.axis("off")
    fig.suptitle(title)
    plt.show()

## Build the dataset

Finds the `.avi` files under `data/Batch*/` for this split.

In [ ]:
dataset = EchoNetLVHDataset(data_dir=str(DATA_DIR), split="train", clip_len=4)
fpath = dataset.fnames[0]
print(fpath)

## Raw video

Shape is (frames, height, width, channels)

In [ ]:
video = load_video(fpath)
print(video.shape, video.dtype)
show_clip(
    video[:: max(1, len(video) // 4)][:4, ..., ::-1],
    "raw video (4 evenly spaced frames)",
)

## The full `__getitem__` output

Two views, each (C, T, H, W), plus frame-order labels.

In [ ]:
x_i, x_j, t_i, t_j = dataset[0]
print(x_i.shape, x_j.shape, t_i.item(), t_j.item())
show_clip(x_i.permute(1, 2, 3, 0).numpy()[..., ::-1], "view i")
show_clip(x_j.permute(1, 2, 3, 0).numpy()[..., ::-1], "view j")

## Step 4: forward a small batch through SimCLR and compute the pretraining loss

In [ ]:
batch_size = 2
loader = torch.utils.data.DataLoader(
    dataset, batch_size=batch_size, shuffle=True, drop_last=True
)
x_i, x_j, t_i, t_j = [b.to(DEVICE) for b in next(iter(loader))]

encoder = torchvision.models.video.r3d_18(weights=None)
model = SimCLR(
    encoder=encoder, projection_dim=128, n_features=encoder.fc.in_features
).to(DEVICE)

with torch.no_grad():
    h_i, h_j, z_i, z_j, t_hat_i, t_hat_j = model(x_i, x_j)
print(
    "features h:",
    h_i.shape,
    " projections z:",
    z_i.shape,
    " reorder logits:",
    t_hat_i.shape,
)

contrastive = NT_Xent(batch_size, temperature=0.05)(z_i, z_j)
reorder = torch.nn.functional.cross_entropy(
    torch.cat([t_hat_i, t_hat_j]), torch.cat([t_i, t_j])
)
print(
    f"NT-Xent={contrastive.item():.4f}  reorder CE={reorder.item():.4f}  total={(contrastive + reorder).item():.4f}"
)

## Training Loop: First Iteration

#### Copied from ssl_pretraining/main.py

In [ ]:
import argparse
import os
import shutil

import pandas as pd
import torch
import torchvision
import tqdm
from dataset import EchoDataset
from losses import NT_Xent
from model import SimCLR
from utils import seed_worker, set_seed


In [ ]:
data_dir = "./data"  # required, no default -- fill in
output_dir = "./output"  # required, no default -- fill in
model_name = "test-model"  # required, no default -- fill in

n_gpu = 1

batch_size = 2
temperature = 0.05
projection_dim = 128
lr = 0.1
num_epochs = 300
clip_len = 4
sampling_rate = 1

save_freq = 20


In [ ]:
# Create output directory and clean out if already exists
if not os.path.isdir(output_dir):
    os.mkdir(output_dir)

model_dir = os.path.join(output_dir, model_name)

if os.path.isdir(model_dir):
    shutil.rmtree(model_dir)
os.mkdir(model_dir)

history = pd.DataFrame({"epoch": [], "loss": []})
history.to_csv(os.path.join(model_dir, "history.csv"), index=False)


In [ ]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

set_seed(0)

train_dataset = EchoNetLVHDataset(
    data_dir=str(DATA_DIR), split="train", clip_len=clip_len
)
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=n_gpu * batch_size,
    worker_init_fn=seed_worker,
    shuffle=True,
    drop_last=True,
)

encoder = torchvision.models.video.r3d_18(weights=None)
model = SimCLR(
    encoder=encoder, projection_dim=projection_dim, n_features=encoder.fc.in_features
)

model = model.to(device)
print(model)

optimizer = torch.optim.Adam(model.parameters(), lr=lr)

loss_fxn = NT_Xent(batch_size, temperature, world_size=n_gpu)
cls_loss_fxn = torch.nn.CrossEntropyLoss()

model.train()
for epoch in range(1, 20 + 1):
    running_loss = 0.0
    pbar = tqdm.tqdm(
        enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch}"
    )

    for i, batch in pbar:
        x_i, x_j, t_i, t_j = batch
        x_i = x_i.to(device)
        x_j = x_j.to(device)
        t_i = t_i.to(device)
        t_j = t_j.to(device)

        h_i, h_j, z_i, z_j, t_hat_i, t_hat_j = model(x_i, x_j)

        loss = loss_fxn(z_i, z_j) + cls_loss_fxn(
            torch.cat([t_hat_i, t_hat_j]), torch.cat([t_i, t_j])
        )

        # Backward pass
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        pbar.set_postfix({"loss": running_loss / (i + 1)})

    current_metrics = pd.DataFrame({"epoch": [epoch], "loss": [running_loss / (i + 1)]})
    current_metrics.to_csv(
        os.path.join(model_dir, "history.csv"), mode="a", header=False, index=False
    )

    if epoch % save_freq == 0:
        torch.save(
            {
                "weights": model.module.state_dict()
                if isinstance(model, torch.nn.DataParallel)
                else model.state_dict(),
                "optimizer": optimizer.state_dict(),
            },
            os.path.join(model_dir, f"chkpt_epoch-{epoch}.pt"),
        )
